Hay que tener instalado la versión de Java 17. Para ello, ejecutar en el terminal:
```bash
sudo apt install -y openjdk-17-jre openjdk-17-jdk
```

Cuando se ejecute en local, asegurarse de que usamos el kernel con el entorno virtual y en el terminal ejecutar:

```bash
pip install pyspark==4.0.0 requests pillow ipython boto3
sudo apt install tree
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
```

In [2]:
import io
import json
import os
import boto3
from PIL import Image

In [3]:
# Configuración centralizada (debe coincidir con tu Master)
IP_CENTRAL = "192.168.0.20"
BUCKET_NAME = "bacteria-dataset"

In [4]:
def get_s3_client():
    """Devuelve un cliente de boto3 conectado al MinIO de la clase."""
    return boto3.client(
        's3',
        endpoint_url=f"http://{IP_CENTRAL}:9000",
        aws_access_key_id="admin_clase",
        aws_secret_access_key="PasswordClase2026",
        config=boto3.session.Config(signature_version='s3v4')
    )

### Funciones para leer y escribir archivos JSON e imágenes desde S3

In [5]:
def s3_read_json(s3_client, bucket, s3_key):
    """Descarga un archivo JSON desde S3 y lo parsea a un diccionario de Python."""
    response = s3_client.get_object(Bucket=bucket, Key=s3_key)
    return json.loads(response['Body'].read().decode('utf-8'))

def s3_read_image(s3_client, bucket, s3_key):
    """Descarga los bytes de una imagen desde S3 y devuelve un objeto PIL Image."""
    response = s3_client.get_object(Bucket=bucket, Key=s3_key)
    img_bytes = response['Body'].read()
    return Image.open(io.BytesIO(img_bytes))

def s3_write_json(s3_client, bucket, s3_key, data):
    """Sube un diccionario de Python como archivo JSON a S3."""
    json_buffer = io.StringIO()
    json.dump(data, json_buffer, indent=4)
    s3_client.put_object(Bucket=bucket, Key=s3_key, Body=json_buffer.getvalue())

def s3_write_image(s3_client, bucket, s3_key, pil_image, format_img="JPEG"):
    """Sube un objeto PIL Image directamente a S3 sin pasar por disco local."""
    img_buffer = io.BytesIO()
    pil_image.save(img_buffer, format=format_img)
    img_buffer.seek(0)
    s3_client.put_object(Bucket=bucket, Key=s3_key, Body=img_buffer)

In [5]:
def resize_image_and_labels(img, original_labels, target_size=(1024, 1024), original_size=(2048, 2048)):
    """
    Reescala la imagen de la bacteria y recalcula las cajas del JSON proporcionalmente
    para mantener la consistencia en tareas de segmentación/detección.
    """
    # 1. Reducción de resolución de la imagen
    img_resized = img.resize(target_size)
    
    # 2. Cálculo de escalas
    scale_x = target_size[0] / original_size[0]
    scale_y = target_size[1] / original_size[1]
    
    # 3. Ajuste de las etiquetas geométricas
    new_labels = []
    for label in original_labels:
        new_labels.append({
            "id": label["id"],
            "class": label["class"],
            "x": int(label["x"] * scale_x),
            "y": int(label["y"] * scale_y),
            "width": int(label["width"] * scale_x),
            "height": int(label["height"] * scale_y)
        })
        
    return img_resized, new_labels

def crop_bacterial_colony(img, label):
    """Realiza un recorte (crop) de alta calidad de una colonia específica usando su caja original."""
    x, y, w, h = label["x"], label["y"], label["width"], label["height"]
    return img.crop((x, y, x + w, y + h))

In [6]:
def process_image_and_json(row):
    """
    Función principal distribuida. Orquesta la lectura, transformación y
    guardado de datos utilizando almacenamiento centralizado S3/MinIO.
    """
    # Spark nos da el path completo (ej: s3a://bacteria-dataset/AGAR_representative/.../518.json)
    # Extramos el 'Key' relativo interno eliminando el prefijo del protocolo y bucket
    full_path = row['path']
    prefix_to_remove = f"s3a://{BUCKET_NAME}/"
    json_key = full_path.replace(prefix_to_remove, "")
    
    # Derivamos el nombre base y el path de la imagen correspondiente en S3
    filename_base = os.path.basename(json_key).replace(".json", "")
    img_key = json_key.replace(".json", ".jpg")
    
    # Inicializamos el cliente en el Worker
    s3 = get_s3_client()
    
    try:
        # --- FASE 1: LECTURA CENTRALIZADA ---
        data = s3_read_json(s3, BUCKET_NAME, json_key)
        img = s3_read_image(s3, BUCKET_NAME, img_key)
        
        # --- FASE 2: TRANSFORMACIÓN (REDUCCIÓN) ---
        img_resized, new_labels = resize_image_and_labels(img, data.get("labels", []))
        
        # --- FASE 3: TRANSFORMACIÓN (RECORTE) Y GUARDADO DE CROPS ---
        for label in data.get("labels", []):
            colony_crop = crop_bacterial_colony(img, label)
            
            # Ruta de destino en S3 (Ej: output/crops/B.subtilis/13895_1.jpg)
            crop_s3_key = f"output/crops/{label['class']}/{data['sample_id']}_{label['id']}.jpg"
            s3_write_image(s3, BUCKET_NAME, crop_s3_key, colony_crop)
            
        # --- FASE 4: GUARDADO DE RESULTADOS DE SEGMENTACIÓN ---
        s3_write_image(s3, BUCKET_NAME, f"output/segmentation/images/{filename_base}.jpg", img_resized)
        
        data["labels"] = new_labels
        s3_write_json(s3, BUCKET_NAME, f"output/segmentation/labels/{filename_base}.json", data)
        
        return f"Procesado con éxito en S3: {filename_base}"
        
    except Exception as e:
        return f"Error procesando {filename_base}: {str(e)}"

In [7]:
import os
# Cambia esta ruta a la carpeta exacta donde vas a guardar el 'core-site.xml'
# (Por ejemplo, la raíz de tu laboratorio)
os.environ["SPARK_CONF_DIR"] = "/home/laptop/lab_spark"

# Limpiamos cualquier rastro de la sesión de Spark previa en la memoria de Python
import sys
if 'pyspark' in sys.modules:
    try:
        print('Parando sesión de Spark...')
        # Parar la sesión de Spark si está en ejecución
        spark.stop()
    except:
        pass

In [8]:
from pyspark.sql import SparkSession

IP_CENTRAL = "192.168.0.20"

spark = SparkSession.builder \
    .master(f"spark://{IP_CENTRAL}:7077") \
    .config("spark.driver.host", IP_CENTRAL) \
    .appName("BacteriaDatasetProcessing_S3") \
    .getOrCreate()

print("¡Spark Master conectado de forma limpia y nativa!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 21:14:26 WARN Utils: Your hostname, laptop-Creator-M16-A12UC, resolves to a loopback address: 127.0.1.1; using 192.168.0.20 instead (on interface wlo1)
26/06/21 21:14:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/21 21:14:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/21 21:14:37 WARN StandaloneAppClient$ClientEndpoint: Failed to connect to master 192.168.0.20:7077
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtil

Py4JJavaError: An error occurred while calling None.org.apache.spark.sql.classic.SparkSession.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(Constru

In [ ]:
import boto3

# Usamos Python puro en el Driver para listar qué hay que procesar
s3_client = boto3.client(
    's3', 
    endpoint_url=f"http://{IP_CENTRAL}:9000", 
    aws_access_key_id="admin_clase", 
    aws_secret_access_key="PasswordClase2026"
)

response = s3_client.list_objects_v2(Bucket="bacteria-dataset", Prefix="AGAR_representative/")

# Extraemos solo los Keys de los JSON
json_keys = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.json')]

# Paralelizamos la lista de strings (Rutas internas de S3)
paths_rdd = spark.sparkContext.parallelize(json_keys)
print(f"Número de JSONs detectados y distribuidos en el clúster: {paths_rdd.count()}")

In [ ]:
# Tu función 'process_image_and_json' recibirá directamente el json_key string 
# y usará get_s3_client() de boto3 de forma 100% estanca en los hilos de los alumnos.
results = paths_rdd.map(process_image_and_json).collect()

for res in results[:10]:
    print(res)

In [ ]:

# 🌟 RUTA UNIVERSAL CLOUD
# No importa el usuario, ni el sistema operativo, ni el nodo. La ruta es unívoca.
dataset_path = "s3a://bacteria-dataset/AGAR_representative/"

# Lector nativo optimizado de Spark
json_files_df = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.json") \
    .option("recursiveFileLookup", "true") \
    .load(dataset_path) \
    .select("path")

print(f"Número de archivos JSON detectados de forma nativa en MinIO: {json_files_df.count()}")

In [ ]:
# 1. Convertimos el DataFrame de paths a un RDD de filas
# 2. Mapeamos cada fila usando la función orquestadora distribuida
# 3. Ejecutamos la acción .collect() para disparar el procesamiento distribuido
results = json_files_df.rdd.map(process_image_and_json).collect()

# --- FEEDBACK PARA CLASE ---
print("\n--- RESUMEN DEL PROCESAMIENTO DISTRIBUIDO ---")
exitos = sum(1 for res in results if "con éxito" in res)
errores = sum(1 for res in results if "Error" in res)

print(f"Total procesados: {len(results)}")
print(f"✅ Éxitos volcados a MinIO: {exitos}")
print(f"❌ Errores en nodos: {errores}\n")

# Mostrar un desglose de los primeros 10 resultados para verificar visualmente
print("Muestra de resultados:")
for res in results[:10]:
    print(f"  > {res}")

# Detener la sesión de Spark de forma limpia al terminar la práctica
spark.stop()